In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. 选择并加载模型和 Tokenizer
model_name = r"C:\Users\k\Desktop\BaiduSyncdisk\baidu_sync_documents\hf_models\Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 确保模型处于评估模式
model.eval()


In [ ]:

# 2. 准备输入文本
prompt = "你是谁？"
input_ids = tokenizer.encode(prompt, return_tensors="pt") # "pt" 代表 PyTorch tensors

# 3. 执行单步推理（前向传播）
# 我们不需要计算梯度，所以使用 no_grad() 来节省计算资源并避免意外的训练。
with torch.no_grad():
    outputs = model(input_ids)
    # 'outputs' 包含多个部分，我们最关心的是 'logits'
    # logits 的形状通常是 [batch_size, sequence_length, vocab_size]
    # 我们想要的是 *下一个* Token 的预测，所以我们取序列中 *最后一个* Token 的 logits
    next_token_logits = outputs.logits[:, -1, :] # 形状变为 [batch_size, vocab_size]


In [ ]:

# 4. 获取候选 Token 及其分数
# 我们可以获取 Logit 值最高的 K 个 Token

k = 10 # 我们想看 Top 10 的候选者

# (a) 获取 Top-K Logits
top_k_logits, top_k_indices = torch.topk(next_token_logits, k)
top_k_logits, top_k_indices

In [ ]:

# (b) (可选) 将 Logits 转换为概率
# 使用 Softmax 函数将 Logits 转换为概率分布
probabilities = torch.nn.functional.softmax(next_token_logits, dim=-1)
probabilities

In [ ]:
top_k_probs, top_k_indices_probs = torch.topk(probabilities, k) # 这里的 indices 应该和上面一样
top_k_probs, top_k_indices_probs

In [ ]:

# 5. 打印结果
# 将 Token 索引解码回文本
top_k_tokens = tokenizer.convert_ids_to_tokens(top_k_indices[0]) # [0] 是因为 batch_size=1
top_k_tokens

In [ ]:
top_k_tokens_probs = tokenizer.convert_ids_to_tokens(top_k_indices_probs[0])
top_k_tokens_probs

In [ ]:

print(f"输入文本: '{prompt}'")
print("-" * 30)
print(f"模型预测的 Top-{k} 个下一个 Token (基于 Logits):")

for token, logit in zip(top_k_tokens, top_k_logits[0]):
    # 使用 .item() 从 Tensor 中提取 Python 数字
    print(f"  - Token: '{token.replace('Ġ', ' ')}', Logit: {logit.item():.4f}") # 'Ġ' 通常代表空格


In [ ]:
print("-" * 30)
print(f"模型预测的 Top-{k} 个下一个 Token (基于概率):")

for token, prob in zip(top_k_tokens_probs, top_k_probs[0]):
    # 解码单个token为可读文本
    decoded_token = tokenizer.decode([tokenizer.convert_tokens_to_ids(token)])
    print(f"  - Token: '{token}' -> 解码: '{decoded_token}', 概率: {prob.item():.4%}")


In [ ]:

# 6. (可选) 演示如何选择一个 Token 并继续生成
chosen_token_id = top_k_indices[0][0].unsqueeze(0).unsqueeze(0) # 选择 Logit 最高的那个
new_input_ids = torch.cat([input_ids, chosen_token_id], dim=-1)
new_prompt = tokenizer.decode(new_input_ids[0])

print("-" * 30)
print(f"如果选择 Logit 最高的 Token，下一个输入将是: '{new_prompt}'")